# 11 — Logging, Configuration, and Error Boundaries

As programs become more complex, print-based debugging is not enough. Use structured logging, explicit configuration, and clear failure boundaries to make behavior observable and maintainable.

**Level:** Intermediate · **Study time:** 75–120 minutes · **Prerequisites:** Python Basics lessons and earlier Intermediate notebooks. **Runtime:** Python 3.10+ with a Jupyter Python kernel.

## Learning objectives

- Choose useful log levels and messages
- Avoid repeated global logging configuration in notebooks
- Model configuration with dataclasses and validation
- Distinguish domain errors from I/O errors

**Study method:** Predict each code cell's output, run it, alter one input, and explain the result. Restart the kernel and run the notebook in order before moving on. Every runnable example uses the standard library and avoids live network requests.

## Guided lessons with worked examples

The sections below include conceptual explanations, executable code, and reasoning prompts. Think about error handling, state, and design trade-offs, not just syntax.

### 1. Logging levels and reusable loggers

The logging module supports DEBUG, INFO, WARNING, ERROR, and CRITICAL. Libraries should retrieve named loggers rather than configuring the root logger at import time. In notebooks, a dedicated StreamHandler can be added and removed safely.

In [ ]:
import logging

logger = logging.getLogger("course.intermediate")
logger.setLevel(logging.DEBUG)
handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
logger.addHandler(handler)
try:
    logger.info("loaded %d records", 3)
    logger.warning("missing optional field: %s", "postcode")
finally:
    logger.removeHandler(handler)
    handler.close()

**Think about it:** Why are parameterized log messages preferable to eager string formatting?

### 2. Validating configuration

Configuration groups environment-dependent settings in one place. A dataclass makes dependencies explicit; __post_init__ can reject invalid values. Never hardcode credentials in notebooks or commit .env files containing secrets.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Settings:
    retries: int = 3
    timeout_seconds: float = 5.0
    def __post_init__(self):
        if self.retries < 0 or self.timeout_seconds <= 0:
            raise ValueError("invalid settings")

settings = Settings(retries=2)
print(settings)

**Think about it:** Where should a real application load API tokens, and why should they not appear in logs?

### 3. Separating error types

A boundary adapter converts low-level errors into clear application-level failures. Preserve context with raise ... from error when helpful. Avoid swallowing unexpected exceptions, since a hidden failure is difficult to debug.

In [ ]:
class RecordError(Exception):
    """Raised when an input record cannot be processed."""

def parse_count(raw):
    try:
        count = int(raw)
    except (ValueError, TypeError) as error:
        raise RecordError(f"invalid count: {raw!r}") from error
    if count < 0:
        raise RecordError("count must not be negative")
    return count

for text in ["5", "-2", "many"]:
    try:
        print(parse_count(text))
    except RecordError as exc:
        print(type(exc).__name__, exc)

**Think about it:** What is the advantage of preserving the original exception as the cause?

### 4. Redacting sensitive data

Use log messages that reveal operational context without exposing credentials or personally sensitive content. Prefer identifiers or counts over raw user data. This simple example constructs a deliberately redacted context.

In [ ]:
def safe_context(event: str, request: dict) -> dict:
    return {
        "event": event,
        "has_token": bool(request.get("token")),
        "field_count": len(request),
    }

request = {"token": "example-secret-never-log", "page": 2}
print(safe_context("fetch", request))

**Think about it:** Would recording a raw authorization header be safe in debug output?

## Independent exercises

For each question, write a short plan, implement a solution, and add one normal and one edge-case test. Matching topic workbooks contain additional practice.

**Exercise 1.** Design a dataclass holding a positive page size and a retry count.

In [ ]:
# Exercise 1
# Implement your solution here, and add tests.

**Exercise 2.** Log a record count at INFO and a missing optional field at WARNING.

In [ ]:
# Exercise 2
# Implement your solution here, and add tests.

**Exercise 3.** Wrap invalid numeric input in a domain-specific exception with raise from.

In [ ]:
# Exercise 3
# Implement your solution here, and add tests.

**Exercise 4.** Write a logging checklist explaining what must not be included in logs.

In [ ]:
# Exercise 4
# Implement your solution here, and add tests.

## Common pitfalls

- Repeatedly calling logging.basicConfig can behave unexpectedly in notebooks.
- Never log tokens, passwords, or private user data.
- Catch narrow exceptions; do not turn every error into a generic success value.

## Final reflection

1. What new Python abstraction did you use in this lesson?
2. Why is it better than an equivalent beginner approach in some circumstances?
3. What failure mode should a reviewer look for?
4. Can the entire notebook run from a fresh kernel?

**Next:** Continue to the following numbered lesson and complete the corresponding independent exercise pack.